In [1]:
import sys
from google import genai
sys.path.append('../')
from src.load_data import *
from src.agents import *
from src.fine_tuned_agents_phase3 import *

In [2]:
testing_data = load_data_from_csv('../data/test_sent_emo.csv')
print(testing_data.head())

Data loaded successfully from ../data/test_sent_emo.csv
   Sr No.                                          Utterance Speaker  \
0       1  Why do all youre coffee mugs have numbers on ...    Mark   
1       2  Oh. Thats so Monica can keep track. That way ...  Rachel   
2       3                                       Y'know what?  Rachel   
3      19                     Come on, Lydia, you can do it.    Joey   
4      20                                              Push!    Joey   

    Emotion Sentiment  Dialogue_ID  Utterance_ID  Season  Episode  \
0  surprise  positive            0             0       3       19   
1     anger  negative            0             1       3       19   
2   neutral   neutral            0             2       3       19   
3   neutral   neutral            1             0       1       23   
4       joy  positive            1             1       1       23   

      StartTime       EndTime  
0  00:14:38,127  00:14:40,378  
1  00:14:40,629  00:14:47,385  


In [3]:
testing_data.drop_duplicates(inplace=True)

In [4]:
# Create a unique key like "102_5" (Dialogue_ID + Utterance_ID)
testing_data['Recognition_ID'] = (
    testing_data['Dialogue_ID'].astype(str) + "_" + 
    testing_data['Utterance_ID'].astype(str) + "_" +
    testing_data['Season'].astype(str) + "_" +
    testing_data['Episode'].astype(str)
)

print(testing_data[['Dialogue_ID', 'Utterance_ID', 'Recognition_ID']].head())

   Dialogue_ID  Utterance_ID Recognition_ID
0            0             0       0_0_3_19
1            0             1       0_1_3_19
2            0             2       0_2_3_19
3            1             0       1_0_1_23
4            1             1       1_1_1_23


In [5]:
def preprocess_test_data(testing_data):
    df = testing_data.copy()
    # Ensure chronological order
    df = df.sort_values(['Dialogue_ID', 'Utterance_ID'])
    
    scenes = []
    for diag_id, group in df.groupby('Dialogue_ID'):
        # 1. The 'Public' data we give to agents (No labels!)
        agent_view = group[['Utterance', 'Speaker', 'Recognition_ID']].to_dict(orient='records')
        
        # 2. The 'Private' data for F1 calculation
        ground_truth = group[['Recognition_ID', 'Emotion', 'Sentiment']].to_dict(orient='records')
        
        scene_data = {
            "dialogue_id": diag_id,
            "utterances": agent_view,  # Sent to Agents
            "ground_truth": ground_truth, # Saved for Evaluation
            "total_turns": len(group)
        }
        scenes.append(scene_data)
    return scenes

preprocessed_test_data = preprocess_test_data(testing_data)
preprocessed_test_data_df = pd.DataFrame(preprocessed_test_data)
print(preprocessed_test_data_df.head())
print(preprocessed_test_data[0])
# Check exactly what the agents are seeing
print(f"Available keys in your data: {preprocessed_test_data[0]['utterances'][0].keys()}")

   dialogue_id                                         utterances  \
0            0  [{'Utterance': 'Why do all youre coffee mugs ...   
1            1  [{'Utterance': 'Come on, Lydia, you can do it....   
2            2  [{'Utterance': 'Okay.', 'Speaker': 'Ross', 'Re...   
3            3  [{'Utterance': 'Oh, okay, I get it.', 'Speaker...   
4            4  [{'Utterance': 'Ohh!', 'Speaker': 'Phoebe', 'R...   

                                        ground_truth  total_turns  
0  [{'Recognition_ID': '0_0_3_19', 'Emotion': 'su...            3  
1  [{'Recognition_ID': '1_0_1_23', 'Emotion': 'ne...            8  
2  [{'Recognition_ID': '2_0_5_16', 'Emotion': 'ne...           11  
3  [{'Recognition_ID': '3_0_5_15', 'Emotion': 'ne...            7  
4  [{'Recognition_ID': '4_0_4_14', 'Emotion': 'su...            9  
{'dialogue_id': 0, 'utterances': [{'Utterance': 'Why do all you\x92re coffee mugs have numbers on the bottom?', 'Speaker': 'Mark', 'Recognition_ID': '0_0_3_19'}, {'Utterance': '

In [ ]:
def run_phase3_council(scene_obj):
    """
    Processes a single scene using the keys present in your data.
          All utterances within a scene are processed in parallel.
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    
    dialogue_id = scene_obj["dialogue_id"]
    utterances = scene_obj["utterances"]
    ground_truth_map = {
        item['Recognition_ID']: {
            'Emotion': item['Emotion'], 
            'Sentiment': item['Sentiment']
        } 
        for item in scene_obj.get("ground_truth", [])
    }

    # Format script for Level 1 agents
    scene_script = "\n".join([f"{u['Speaker']}: {u['Utterance']}" for u in utterances])

    # -- LEVEL 1: GLOBAL CONTEXT --
    global_context = groq_llm_call(
        prompt=f"{context_manager_prompt}\n\nScene:\n{scene_script}", 
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        api_key=context_manager_api
    )
    social_graph = groq_llm_call(
        prompt=f"{relational_graph_prompt}\n\nScene:\n{scene_script}", 
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        api_key=relational_graph_api
    )

    def process_utterance(idx, u):
        """Process a single utterance with all its agents."""
        try:
            target_text = u.get('Utterance', "")
            speaker = u.get('Speaker', "Unknown")
            rec_id = u.get('Recognition_ID', "unknown_id")
            actual_labels = ground_truth_map.get(rec_id, {"Emotion": "neutral", "Sentiment": "neutral"})
            actual_emotion = actual_labels["Emotion"]

            previous_turn = utterances[idx - 1] if idx > 0 else None
            previous_text = previous_turn.get('Utterance', "") if previous_turn else ""
            previous_speaker = previous_turn.get('Speaker', "Unknown") if previous_turn else "Unknown"

            # Run all agent analysis in parallel for this utterance
            with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
                # Submit all agent calls
                f_profile = executor.submit(call_tuned_profiler, target_text, social_graph)
                f_sentiment = executor.submit(call_tuned_sentiment, target_text)
                f_shift = None
                f_dynamics = None
                
                if previous_turn is not None:
                    f_shift = executor.submit(
                        call_emotional_shift,
                        previous_text,
                        previous_speaker,
                        target_text,
                        speaker,
                        global_context
                    )
                
                # Collect profile and sentiment first
                profile_report = f_profile.result()
                sentiment_report = f_sentiment.result()
                
                # Emotional shift result
                if f_shift is not None:
                    emotional_shift_report = f_shift.result()
                else:
                    emotional_shift_report = "[SHIFT: FALSE - CONTINUITY MAINTAINED]"
                
                # Submit social dynamics (needs profile_report)
                f_dynamics = executor.submit(
                    call_social_dynamics, 
                    target_text, 
                    profile_report, 
                    social_graph
                )
                dynamics_report = f_dynamics.result()

            # Final aggregator (sequential, needs all reports)
            final_prediction_raw = call_council_aggregator(
                rec_id,
                target_text,
                global_context,
                profile_report,
                sentiment_report,
                dynamics_report,
                emotional_shift_report
            )

            return {
                "Dialogue_ID": dialogue_id,
                "Recognition_ID": rec_id,
                "Speaker": speaker,
                "Utterance": target_text,
                "Predicted_Emotion_Raw": final_prediction_raw,
                "Actual_Emotion": actual_emotion
            }
        except Exception as e:
            raise Exception(f"Error processing utterance {idx} (rec_id={u.get('Recognition_ID')}): {str(e)}")

    # -- LEVEL 2 & 3: PROCESS ALL UTTERANCES IN PARALLEL --
    scene_results = []
    max_workers_utterance = min(5, len(utterances))  # Parallelize across utterances
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers_utterance) as executor:
        futures = {
            executor.submit(process_utterance, idx, u): idx 
            for idx, u in enumerate(utterances)
        }
        
        for future in concurrent.futures.as_completed(futures):
            try:
                result = future.result()
                scene_results.append(result)
            except Exception as e:
                raise e
    
    # Sort by original index to maintain order
    scene_results.sort(key=lambda x: utterances[next(i for i, u in enumerate(utterances) if u['Recognition_ID'] == x['Recognition_ID'])]['Recognition_ID'])
    
    return scene_results

<>:2: SyntaxWarning: invalid escape sequence '\ '
<>:2: SyntaxWarning: invalid escape sequence '\ '
C:\Users\deepa\AppData\Local\Temp\ipykernel_23656\217702102.py:2: SyntaxWarning: invalid escape sequence '\ '
  """


In [ ]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

output_file = "../logs/council_phase3_3_results.csv"

# 1. Identify which scenes are already done
processed_ids = set()
if os.path.exists(output_file):
    try:
        existing_df = pd.read_csv(output_file)
        if not existing_df.empty:
            processed_ids = set(existing_df['Dialogue_ID'].unique())
            print(f"Found {len(processed_ids)} scenes already processed. Skipping them...")
    except Exception as e:
        print(f"Could not read existing results: {e}")

all_final_results = []
stop_processing = threading.Event()
csv_lock = threading.Lock()

def process_scene_worker(scene, index, total):
    """Worker function to process a single scene in parallel."""
    try:
        scene_id = scene['dialogue_id']
        print(f"[Worker] Processing Scene {index+1}/{total} (ID: {scene_id})...")
        
        scene_out = run_phase3_council(scene)
        return (scene_id, scene_out, None)
    except Exception as e:
        error_msg = str(e).lower()
        if "api key" in error_msg or "unauthorized" in error_msg or "401" in error_msg:
            return (None, None, ("AUTH_ERROR", str(e)))
        elif "429" in error_msg or "resource exhausted" in error_msg:
            return (None, None, ("RATE_LIMIT", str(e)))
        else:
            return (None, None, ("PROCESS_ERROR", str(e)))

# 2. Filter unprocessed scenes
scenes_to_process = [
    (i, scene) for i, scene in enumerate(preprocessed_test_data)
    if scene['dialogue_id'] not in processed_ids
]

print(f"Total unprocessed scenes: {len(scenes_to_process)}")

# 3. Process scenes in parallel (3 workers to respect API rate limits)
max_workers = 3
rate_limit_hit = False

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {
        executor.submit(process_scene_worker, scene, idx, len(scenes_to_process)): (idx, scene)
        for idx, scene in scenes_to_process
    }
    
    completed = 0
    for future in as_completed(futures):
        if stop_processing.is_set():
            break
            
        idx, scene = futures[future]
        scene_id, scene_out, error = future.result()
        
        if error:
            error_type, error_msg = error
            if error_type == "AUTH_ERROR":
                print(f"❌ Authentication/config error at Scene {scene_id}. Fix OpenRouter key/env and rerun.")
                print(error_msg)
                stop_processing.set()
                break
            elif error_type == "RATE_LIMIT":
                print(f"⏱️ Rate limit exhausted at Scene {scene_id}. Stopping gracefully.")
                print(error_msg)
                stop_processing.set()
                rate_limit_hit = True
                break
            else:
                print(f"⚠️ Error in Scene {scene_id}: {error_msg}")
                continue
        
        # Write scene results to CSV immediately (thread-safe)
        if scene_out:
            
            with csv_lock:
                new_df = pd.DataFrame(scene_out)
                new_df.to_csv(output_file, mode='a', index=False, header=not os.path.exists(output_file))
                all_final_results.extend(scene_out)
                print(f"✓ Scene {scene_id} saved to {output_file}")
        
        completed += 1
        print(f"Progress: {completed}/{len(scenes_to_process)} scenes completed")

if rate_limit_hit:
    print("\n🛑 PAUSED: Rate limit hit. Wait before re-running.")
else:
    print("\n✅ SESSION FINISHED. All unprocessed scenes in this batch are done!")

Found 27 scenes already processed. Skipping them...
Total unprocessed scenes: 253
[Worker] Processing Scene 26/253 (ID: 25)...
[Worker] Processing Scene 29/253 (ID: 28)...
[Worker] Processing Scene 30/253 (ID: 29)...


In [ ]:
import pandas as pd
import json
import re
from sklearn.metrics import classification_report, f1_score, confusion_matrix

def generate_erc_report(csv_path):
    # 1. Load the results
    df = pd.read_csv(csv_path)
    
    def extract_label(raw_string):
        """Extracts 'predicted_emotion' from the Aggregator's JSON string."""
        if pd.isna(raw_string):
            return "neutral"
        try:
            clean_json = raw_string.replace('""', '"')
            match = re.search(r'\{.*\}', clean_json, re.DOTALL)
            if match:
                data = json.loads(match.group())
                return data.get("predicted_emotion", "neutral").lower().strip()
            return "neutral"
        except Exception:
            return "neutral"

    # 2. Clean and Align Labels
    df['Predicted_Clean'] = df['Predicted_Emotion_Raw'].apply(extract_label)
    df['Actual_Clean'] = df['Actual_Emotion'].str.lower().str.strip()

    labels = sorted(df['Actual_Clean'].unique())

    print("\n" + "="*60)
    print("      MULTI-AGENT COUNCIL: CLASSIFICATION REPORT")
    print("="*60)
    
    report = classification_report(
        df['Actual_Clean'], 
        df['Predicted_Clean'], 
        labels=labels,
        digits=4
    )
    print(report)

    wf1 = f1_score(df['Actual_Clean'], df['Predicted_Clean'], average='weighted')
    print(f"\nOVERALL WEIGHTED F1 SCORE: {wf1:.4f}")
    print("="*60)

    return df

# Usage:
report_df = generate_erc_report("../logs/council_phase3_3_results.csv")


      MULTI-AGENT COUNCIL: CLASSIFICATION REPORT
              precision    recall  f1-score   support

       anger     0.5185    0.4242    0.4667        33
     disgust     0.5000    0.5000    0.5000         6
        fear     0.1724    0.8333    0.2857         6
         joy     0.4459    0.8250    0.5789        40
     neutral     0.8261    0.5135    0.6333       111
     sadness     0.4375    0.4375    0.4375        16
    surprise     0.6364    0.3500    0.4516        20

    accuracy                         0.5431       232
   macro avg     0.5053    0.5548    0.4791       232
weighted avg     0.6483    0.5431    0.5586       232


OVERALL WEIGHTED F1 SCORE: 0.5586
